# AI Job Agent — PHASE 7: LLM Explanation & Improvement Suggestions

**Goal:** Take the final ranked jobs selected by Phase 6 and generate grounded, candidate-facing explanations using GPT-5.6 Luna. Each explanation describes why the job matches, what skills and qualifications are matched or missing, and what learning areas can improve the candidate's fit.

**Single-job scope:** Each job is processed independently using only its Phase 5/6 matching evidence. Phase 7 does not perform new matching or ranking.

**Phase 6 input:** `final_top_jobs.json`

**Phase 7 output:** `phase7_output.json`

**LLM rules:** The model must use only the provided evidence, must not invent candidate information or job requirements, must not recalculate or modify the Phase 6 match score, and must not infer missing information or experience gaps without explicit evidence. If information is unavailable, it must not guess.



## 1. ***Setup***

In [51]:

import json
from pathlib import Path
from typing import List

from openai import OpenAI
from pydantic import BaseModel, Field
from google.colab import userdata
from google.colab import files

# Load API key
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

assert OPENAI_API_KEY, "OPENAI_API_KEY was not found in Colab Secrets."

client = OpenAI(api_key=OPENAI_API_KEY)

print("Setup completed successfully.")

Setup completed successfully.


## 2. Upload Phase 6 Output
Uploads final_top_jobs.json generated by Phase 6 as the input for Phase 7

In [52]:


from google.colab import files as _colab_files

print("Upload final_top_jobs.json from Phase 6")
uploaded_p6 = _colab_files.upload()

P6_DIR = Path("/content/phase6")
P6_DIR.mkdir(exist_ok=True)

for name, content in uploaded_p6.items():
    (P6_DIR / name).write_bytes(content)

p6_file = P6_DIR / "final_top_jobs.json"

assert p6_file.exists(), \
    "final_top_jobs.json not found among uploaded files."

print("Saved Phase 6 input.")

Upload final_top_jobs.json from Phase 6


Saving final_top_jobs.json to final_top_jobs (1).json
Saved Phase 6 input.


# 3. Input Validation & Data Preparationld
Validates the Phase 6 input and ensures the required job matching fields are available for processing.


In [39]:
# Input Validation & Data Preparation
# Load Phase 6 output

with open(p6_file, "r", encoding="utf-8") as f:
    phase6_data = json.load(f)

assert "candidate_id" in phase6_data
assert "jobs" in phase6_data
assert isinstance(phase6_data["jobs"], list)

print("Phase 6 output loaded successfully.")
print("Candidate ID:", phase6_data["candidate_id"])
print("Number of jobs:", len(phase6_data["jobs"]))



Phase 6 output loaded successfully.
Candidate ID: e07ca2443daccbb5
Number of jobs: 1


# 3.1 Prepare Phase 6 Evidence
Structures the Phase 6 matching results into clean evidence that can be provided to the LLM.

In [40]:
# Prepare Phase 6 evidence
# 3.1 Prepare Phase 6 Evidence

jobs_evidence = []

for job in phase6_data["jobs"]:
    evidence = {
        "job_id": job["job_id"],
        "title": job["title"],
        "company": job["company"],
        "location": job["location"],
        "job_description": job["job_description"],
        "match_score": job["match_score"],
        "scoring_confidence": job["scoring_confidence"],
        "matched_core_skills": job["matched_core_skills"],
        "missing_core_skills": job["missing_core_skills"],
        "matched_qualifications": job["matched_qualifications"],
        "missing_qualifications": job["missing_qualifications"],
        "matched_education_fields": job["matched_education_fields"],
        "missing_education_fields": job["missing_education_fields"]
    }

    jobs_evidence.append(evidence)

print("Phase 6 evidence prepared successfully.")
print("Number of jobs:", len(jobs_evidence))


Phase 6 evidence prepared successfully.
Number of jobs: 1


# 3.2 Define Pydantic Output Schema
Defines the structured output schema used to ensure consistent and validated LLM responses for each job.

In [41]:
# 3.2 Pydantic Output Schema

class ImprovementSuggestion(BaseModel):
    area: str
    suggestion: str


class JobExplanation(BaseModel):
    job_id: str
    match_score: float
    why_match: str
    matched_skills: List[str]
    missing_skills: List[str]
    matched_qualifications: List[str]
    missing_qualifications: List[str]
    matched_education: List[str]
    missing_education: List[str]
    improvement_suggestions: List[ImprovementSuggestion]
    print("Pydantic schema created successfully.")

Pydantic schema created successfully.


# 4. Prompt Engineering

Defines the grounding rules and job-specific instructions used to generate accurate explanations.



In [42]:
# 4. Prompt Engineering

SYSTEM_PROMPT = """
You are the LLM Explanation Agent in an AI Job Matching system.

Your role is to explain the matching results generated by the previous
matching and ranking phases.

The provided evidence comes only from Phase 5 and Phase 6.
Use this evidence as the single source of truth.

Your explanation must:

1. Explain why the job matches the candidate based only on the provided
   matching evidence.

2. Clearly identify the skills that matched.

3. Clearly identify the skills that are missing.

4. Explain matched and missing qualifications using only the provided data.

5. Explain matched and missing education fields using only the provided data.

6. Provide improvement suggestions that are directly related to the
   missing skills, qualifications, or education fields.

7. Preserve the match score exactly as provided by Phase 6.

8. Keep the explanation specific to the current job.

Strict grounding rules:

- Do not invent candidate information.
- Do not invent job requirements.
- Do not add skills that are not present in the provided evidence.
- Do not assume missing information means the candidate does not have it.
- Do not recalculate or modify the match score.
- Do not perform a new matching or ranking process.
- Do not introduce information from external sources.
- Do not infer an experience gap unless explicit evidence for it is provided.
- Improvement suggestions must be grounded in the provided missing evidence.

If information is not available in the evidence, do not fabricate it.

Return only the requested structured output.
"""
# Build job prompt

def build_job_prompt(evidence):
    return f"""
Generate the explanation for this job using only the following Phase 6 evidence:

{json.dumps(evidence, ensure_ascii=False, indent=2)}
"""


# Test prompt construction

test_prompt = build_job_prompt(jobs_evidence[0])

print("Prompt created successfully.")
print("Number of jobs available:", len(jobs_evidence))
print(test_prompt[:3000])


Prompt created successfully.
Number of jobs available: 1

Generate the explanation for this job using only the following Phase 6 evidence:

{
  "job_id": "17a96e147108d5bc",
  "title": "Big Data and AI Manager",
  "company": "Confidential",
  "location": {
    "city": "Riyadh",
    "country": "Saudi Arabia"
  },
  "job_description": "Responsibilities Develop and oversee the strategy for big data and AI initiatives, ensuring alignment with organizational goals and objectives. Manage the design, development, and implementation of big data and AI solutions to drive business insights and operational efficiency. Collaborate with cross-functional teams to identify opportunities where big data and AI technologies can add value and solve complex problems. Oversee the collection, storage, and governance of large datasets, ensuring compliance with data security and privacy regulations. Collaborate with stakeholders to define cases, KPIs, and success metrics for big data and AI projects. Ensure c

# 5. GPT-5.6 Luna — Structured Output
Processes each job independently using GPT-5.6 Luna and generates a structured explanation.

In [43]:
 # 5. GPT-5.6 Luna — Structured Output

llm_outputs = []

for job in jobs_evidence:
    response = client.responses.parse(
        model="gpt-5.6-luna",
        instructions=SYSTEM_PROMPT,
        input=build_job_prompt(job),
        text_format=JobExplanation
    )

    job_explanation = response.output_parsed

    llm_outputs.append(job_explanation)

print("GPT-5.6 Luna processing completed successfully.")
print("Number of jobs processed:", len(llm_outputs))

GPT-5.6 Luna processing completed successfully.
Number of jobs processed: 1


# 6. Validate Output
Checks that the LLM output remains consistent with the original Phase 6 matching evidence

In [15]:
# 6. Validate Output

validated_outputs = []

for phase6_job, llm_output in zip(jobs_evidence, llm_outputs):

    def normalize_items(items):
        return {
            str(item).strip().lower()
            for item in items
        }

    # Validate job ID
    assert llm_output.job_id == phase6_job["job_id"], \
        "job_id was changed by the LLM."

    # Validate match score
    assert llm_output.match_score == phase6_job["match_score"], \
        "match_score was changed by the LLM."

    # Validate matched skills
    phase6_matched_skills = normalize_items(
        phase6_job["matched_core_skills"]
    )

    llm_matched_skills = normalize_items(
        llm_output.matched_skills
    )

    assert llm_matched_skills.issubset(
        phase6_matched_skills
    ), "LLM introduced an unsupported matched skill."

    # Validate missing skills
    phase6_missing_skills = normalize_items(
        phase6_job["missing_core_skills"]
    )

    llm_missing_skills = normalize_items(
        llm_output.missing_skills
    )

    assert llm_missing_skills.issubset(
        phase6_missing_skills
    ), "LLM introduced an unsupported missing skill."

    # Validate matched qualifications
    phase6_matched_qualifications = normalize_items(
        phase6_job["matched_qualifications"]
    )

    llm_matched_qualifications = normalize_items(
        llm_output.matched_qualifications
    )

    assert llm_matched_qualifications.issubset(
        phase6_matched_qualifications
    ), "LLM introduced an unsupported matched qualification."

    # Validate missing qualifications
    phase6_missing_qualifications = normalize_items(
        phase6_job["missing_qualifications"]
    )

    llm_missing_qualifications = normalize_items(
        llm_output.missing_qualifications
    )

    assert llm_missing_qualifications.issubset(
        phase6_missing_qualifications
    ), "LLM introduced an unsupported missing qualification."

    # Validate matched education
    phase6_matched_education = normalize_items(
        phase6_job["matched_education_fields"]
    )

    llm_matched_education = normalize_items(
        llm_output.matched_education
    )

    assert llm_matched_education.issubset(
        phase6_matched_education
    ), "LLM introduced an unsupported matched education field."

    # Validate missing education
    phase6_missing_education = normalize_items(
        phase6_job["missing_education_fields"]
    )

    llm_missing_education = normalize_items(
        llm_output.missing_education
    )

    assert llm_missing_education.issubset(
        phase6_missing_education
    ), "LLM introduced an unsupported missing education field."

    validated_outputs.append(llm_output)

print("Validation completed successfully.")
print("Grounding check: PASSED")
print("Number of jobs validated:", len(validated_outputs))

for output in validated_outputs:
    print("Job ID:", output.job_id)
    print("Match score:", output.match_score)

Validation completed successfully.
Grounding check: PASSED
Number of jobs validated: 1
Job ID: 17a96e147108d5bc
Match score: 30.6971


# 7. Improvement Mapping
Maps missing skills to relevant learning areas to identify potential improvement paths.

In [45]:
# 7. Improvement Mapping

learning_area_map = {
    "ai strategy": "AI Strategy & Planning",
    "big data strategy": "Big Data Strategy & Architecture",
    "communication": "Professional Communication",
    "data governance": "Data Governance & Data Management",
    "data structure": "Data Structures & Data Architecture",
    "leadership": "Leadership",
    "model validation": "Machine Learning Model Validation",
    "project management": "AI/ML Project Management",
    "strategic planning": "Strategic Planning",
    "strategy development": "Strategy Development",
    "vendor management": "Technology Vendor Management"
}

improvement_outputs = []

for job in jobs_evidence:

    learning_areas = []

    for skill in job["missing_core_skills"]:
        normalized_skill = skill.strip().lower()

        learning_area = learning_area_map.get(
            normalized_skill,
            skill
        )

        learning_areas.append({
            "missing_skill": skill,
            "learning_area": learning_area
        })

    improvement_outputs.append({
        "job_id": job["job_id"],
        "learning_areas": learning_areas
    })

print("Improvement areas generated successfully.")
print("Number of jobs processed:", len(improvement_outputs))

for result in improvement_outputs:
    print()
    print("Job ID:", result["job_id"])
    print("Learning areas:", len(result["learning_areas"]))

    for item in result["learning_areas"]:
        print(
            f"- {item['missing_skill']} → "
            f"{item['learning_area']}"
        )

Improvement areas generated successfully.
Number of jobs processed: 1

Job ID: 17a96e147108d5bc
Learning areas: 11
- ai strategy → AI Strategy & Planning
- big data strategy → Big Data Strategy & Architecture
- communication → Professional Communication
- data governance → Data Governance & Data Management
- data structure → Data Structures & Data Architecture
- leadership → Leadership
- model validation → Machine Learning Model Validation
- project management → AI/ML Project Management
- strategic planning → Strategic Planning
- strategy development → Strategy Development
- vendor management → Technology Vendor Management


# 8.  Phase 7 Outputs
Combines the validated explanations and improvement areas into phase7_output.json

In [46]:
# 8. Phase 7 Output

phase7_output = {
    "candidate_id": phase6_data["candidate_id"],
    "created_utc": phase6_data["created_utc"],
    "phase": "Phase 7",
    "jobs": []
}

for phase6_job, llm_output, improvement in zip(
    jobs_evidence,
    validated_outputs,
    improvement_outputs
):

    assert llm_output.job_id == phase6_job["job_id"], \
        "LLM output job_id does not match Phase 6 job_id."

    assert improvement["job_id"] == phase6_job["job_id"], \
        "Improvement output job_id does not match Phase 6 job_id."

    job_output = {
        "job_id": phase6_job["job_id"],
        "title": phase6_job["title"],
        "company": phase6_job["company"],
        "url": phase6_job.get("url"),
        "location": phase6_job["location"],
        "match_score": phase6_job["match_score"],
        "scoring_confidence": phase6_job["scoring_confidence"],
        "why_match": llm_output.why_match,
        "matched_skills": llm_output.matched_skills,
        "missing_skills": llm_output.missing_skills,
        "matched_qualifications": llm_output.matched_qualifications,
        "missing_qualifications": llm_output.missing_qualifications,
        "matched_education": llm_output.matched_education,
        "missing_education": llm_output.missing_education,
        "learning_areas": improvement["learning_areas"]
    }

    phase7_output["jobs"].append(job_output)


OUTPUT_PATH = Path("/content/phase7_output.json")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        phase7_output,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Phase 7 output created successfully.")
print("Number of jobs:", len(phase7_output["jobs"]))
print("Saved to:", OUTPUT_PATH)

Phase 7 output created successfully.
Number of jobs: 1
Saved to: /content/phase7_output.json


# 9. Final Output Validation
Performs a final schema and data-type check to ensure the Phase 7 output is complete and valid.


In [47]:
# 9. Final Output Validation

required_output_fields = [
    "job_id",
    "title",
    "company",
    "url",
    "location",
    "match_score",
    "scoring_confidence",
    "why_match",
    "matched_skills",
    "missing_skills",
    "matched_qualifications",
    "missing_qualifications",
    "matched_education",
    "missing_education",
    "learning_areas"
]

assert "candidate_id" in phase7_output
assert "created_utc" in phase7_output
assert "phase" in phase7_output
assert phase7_output["phase"] == "Phase 7"

assert "jobs" in phase7_output
assert isinstance(phase7_output["jobs"], list)

for job in phase7_output["jobs"]:

    for field in required_output_fields:
        assert field in job, f"Missing output field: {field}"

    assert isinstance(job["job_id"], str)
    assert isinstance(job["title"], str)
    assert isinstance(job["company"], str)

    assert isinstance(job["location"], dict)
    assert "city" in job["location"]
    assert "country" in job["location"]

    assert isinstance(job["match_score"], (int, float))
    assert isinstance(job["scoring_confidence"], (int, float))

    assert isinstance(job["why_match"], str)
    assert len(job["why_match"].strip()) > 0

    assert isinstance(job["matched_skills"], list)
    assert isinstance(job["missing_skills"], list)

    assert isinstance(job["matched_qualifications"], list)
    assert isinstance(job["missing_qualifications"], list)

    assert isinstance(job["matched_education"], list)
    assert isinstance(job["missing_education"], list)

    assert isinstance(job["learning_areas"], list)

print("Final output validation completed successfully.")
print("Phase 7 output is valid and ready for Phase 8.")
print("Number of validated jobs:", len(phase7_output["jobs"]))

Final output validation completed successfully.
Phase 7 output is valid and ready for Phase 8.
Number of validated jobs: 1


# 10. Data Dictionary

Documents the fields and structure of the Phase 7 output for clear handoff and future use

In [48]:
# 10. Data Dictionary

DATA_DICTIONARY = """
# Phase 7 Outputs — Data Dictionary

## phase7_output.json

| Field | Description |
|---|---|
| candidate_id | Identifier of the candidate |
| created_utc | UTC timestamp when the Phase 7 output was created |
| phase | Phase identifier |
| job_id | Unique identifier of the job |
| title | Job title |
| company | Company name |
| url | Job posting URL |
| location | Job location |
| match_score | Match score generated by Phase 6 |
| scoring_confidence | Confidence of the Phase 6 matching score |
| why_match | LLM-generated explanation of why the job matches |
| matched_skills | Skills matched by Phase 6 |
| missing_skills | Skills identified as missing by Phase 6 |
| matched_qualifications | Qualifications matched by Phase 6 |
| missing_qualifications | Qualifications identified as missing by Phase 6 |
| matched_education | Education fields matched by Phase 6 |
| missing_education | Education fields identified as missing by Phase 6 |
| learning_areas | Learning areas mapped from missing skills |
"""

print("Data dictionary created successfully.")

Data dictionary created successfully.


#11. Persist Phase 7 Outputs
Saves the final JSON output and Data Dictionary into the Phase 7 output directory.


In [50]:
# 11. Persist Phase 7 Outputs

import shutil

phase7_dir = Path("/content/phase7_outputs")
phase7_dir.mkdir(exist_ok=True)

# Save Phase 7 JSON
phase7_json_path = phase7_dir / "phase7_output.json"

with open(phase7_json_path, "w", encoding="utf-8") as f:
    json.dump(
        phase7_output,
        f,
        ensure_ascii=False,
        indent=2
    )

# Save Data Dictionary
data_dictionary_path = phase7_dir / "data_dictionary.txt"

with open(data_dictionary_path, "w", encoding="utf-8") as f:
    f.write(DATA_DICTIONARY)

print("Phase 7 outputs persisted successfully.")
print("Output directory:", phase7_dir)

Phase 7 outputs persisted successfully.
Output directory: /content/phase7_outputs


In [ ]:
# 12. Create Phase 7 ZIP Package

import json
import shutil
from pathlib import Path
from google.colab import _message
from google.colab import files

# Create Phase 7 package directory
package_dir = Path("/content/Phase_7_package")
package_dir.mkdir(exist_ok=True)

# Get the current Colab notebook
response = _message.blocking_request(
    "get_ipynb",
    timeout_sec=10
)

notebook = response["ipynb"]

# Save the current notebook
notebook_path = package_dir / "Phase 7.ipynb"

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(
        notebook,
        f,
        ensure_ascii=False,
        indent=2
    )

# Copy Phase 7 output
phase7_output_path = package_dir / "phase7_output.json"

shutil.copy2(
    "/content/phase7_outputs/phase7_output.json",
    phase7_output_path
)

# Copy Data Dictionary
data_dictionary_path = package_dir / "data_dictionary.txt"

shutil.copy2(
    "/content/phase7_outputs/data_dictionary.txt",
    data_dictionary_path
)

# Create ZIP
zip_path = shutil.make_archive(
    "/content/Phase 7",
    "zip",
    package_dir
)

zip_file = Path(zip_path)

# Final checks
assert notebook_path.exists(), "Phase 7 notebook was not created."
assert phase7_output_path.exists(), "phase7_output.json was not found."
assert data_dictionary_path.exists(), "data_dictionary.txt was not found."
assert zip_file.exists(), "Phase 7 ZIP was not created."
assert zip_file.stat().st_size > 0, "Phase 7 ZIP is empty."

print("Phase 7 ZIP package created successfully.")
print()
print("Package contents:")
print("- Phase 7.ipynb")
print("- phase7_output.json")
print("- data_dictionary.txt")
print()
print("ZIP path:", zip_file)
print(
    "ZIP size:",
    f"{zip_file.stat().st_size / (1024**2):.2f} MB"
)

files.download(zip_file)

### Next step — PHASE 8

**Phase 8  use the `phase7_output.json` generated by Phase 7 as its input.**

The Phase 7 output contains:
- Job information
- Match score
- Why the job matches
- Matched and missing skills
- Matched and missing qualifications
- Matched and missing education
- Learning areas